## Шаг 0. Клонирование репозитория (Colab / Kaggle)

На Colab и Kaggle локальной копии репо нет — первой ячейкой клонируем его и добавляем `src/` в `sys.path`, чтобы `from gp1.env import ...` работал. Локально ячейка — no-op.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/DKazhekin/ITMO_Speech_Recognition_Course.git"
REPO_NAME = "ITMO_Speech_Recognition_Course"

_in_colab = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ
_in_kaggle = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if _in_colab or _in_kaggle:
    repo_dir = Path(REPO_NAME)
    if not repo_dir.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    gp1_src = (repo_dir / "group-projects/gp1/src").resolve()
    if str(gp1_src) not in sys.path:
        sys.path.insert(0, str(gp1_src))
    print(f"Repo cloned, gp1 src on sys.path: {gp1_src}")
else:
    print("Local run — repo already checked out.")

# 06. KenLM beam rescore — улучшить greedy через LM-fusion

## Что делает этот ноутбук

Берёт **любой** CharVocab-чекпоинт (QuartzNet, CRDNN, Efficient Conformer),
обучает 4-gram KenLM на синтетическом корпусе русских чисел,
затем сравнивает greedy-декодирование и beam-search с LM-fusion по метрике CER.

In [ ]:
import sys
import os

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), "..", "..", "src"))

from gp1.env import setup_environment
paths, device = setup_environment()
print(f"Platform OK. Device: {device}")

In [ ]:
import torch
import pandas as pd
from pathlib import Path
from torch.utils.data import DataLoader

from gp1.data.manifest import records_from_csv
from gp1.data.dataset import SpokenNumbersDataset
from gp1.data.collate import collate_fn
from gp1.features.melbanks import LogMelFilterBanks
from gp1.decoding.greedy import greedy_decode
from gp1.decoding.beam_pyctc import BeamSearchDecoder, BeamSearchConfig
from gp1.text.vocab import CharVocab
from gp1.text.normalize import digits_to_words
from gp1.train.metrics import compute_cer, compute_per_speaker_cer

## Шаг 1. Обучить KenLM

In [ ]:
from gp1.lm.build_corpus import build_synthetic_corpus
from gp1.lm.train_kenlm import train_kenlm

corpus_path = paths.ckpt_root.parent / "lm" / "synthetic_corpus.txt"
lm_binary   = paths.ckpt_root.parent / "lm" / "numbers.bin"
corpus_path.parent.mkdir(parents=True, exist_ok=True)

if not corpus_path.exists():
    n_sentences = build_synthetic_corpus(out_path=corpus_path)
    print(f"Corpus: {n_sentences} sentences → {corpus_path}")
else:
    print(f"Corpus already exists: {corpus_path}")

if not lm_binary.exists():
    print("Training KenLM (may take ~2 min)...")
    train_kenlm(corpus_path=corpus_path, out_binary=lm_binary, order=4)
    print(f"LM binary: {lm_binary}")
else:
    print(f"LM binary already exists: {lm_binary}")

## Шаг 2. Загрузить char-vocab чекпоинт

In [ ]:
import json

from gp1.train.checkpoint import load_checkpoint

# Укажите путь к checkpoints/<run>/best.pt от любого CharVocab-чекпоинта
CKPT = Path("FILL_ME_IN")  # <-- замените на реальный путь

if CKPT == Path("FILL_ME_IN"):
    raise RuntimeError(
        "CKPT not set. Replace Path('FILL_ME_IN') with a real checkpoint path, "
        "e.g. Path('_local_runs/01_quartznet_.../best.pt'). "
        "Run one of notebooks 01_quartznet / 01a / 01b / 01c first."
    )

if not CKPT.exists():
    raise FileNotFoundError(f"Checkpoint not found: {CKPT}")

meta = json.loads((CKPT.parent / "meta.json").read_text())
arch = meta.get("arch", "quartznet10x4")
hparams = meta.get("hparams", {})
print(f"arch={arch}, val_cer={meta.get('val_cer')}, epoch={meta.get('epoch')}")

vocab = CharVocab()

# Dispatch по arch из meta.json
if arch == "quartznet10x4":
    from gp1.models.quartznet import QuartzNet10x4
    model = QuartzNet10x4(
        vocab_size=vocab.vocab_size,
        d_model=hparams.get("d_model", 256),
        dropout=0.0,
        subsample_factor=hparams.get("subsample_factor", 2),
    )
elif arch == "crdnn":
    from gp1.models.crdnn import CRDNN
    model = CRDNN(vocab_size=vocab.vocab_size, **{k: hparams[k]
                   for k in ("d_model", "dropout") if k in hparams})
elif arch == "efficient_conformer":
    from gp1.models.efficient_conformer import EfficientConformer
    model = EfficientConformer(vocab_size=vocab.vocab_size,
                                **{k: hparams[k] for k in ("d_model", "dropout") if k in hparams})
else:
    raise ValueError(f"Unknown arch {arch!r}. Поддерживаются: quartznet10x4, crdnn, efficient_conformer.")

meta_loaded = load_checkpoint(CKPT, model)
model = model.to(device).eval()
print(f"Model loaded: {arch}")

## Шаг 3. Greedy + beam+LM сравнение на dev

In [ ]:
HP_FIXED = dict(samplerate=16000, n_mels=80, n_fft=512, hop_length=160, win_length=400)

dev_records = records_from_csv(paths.dev_csv, paths.dev_root)
dev_ds = SpokenNumbersDataset(records=dev_records, vocab=vocab,
                               target_samplerate=HP_FIXED["samplerate"], augmenter=None)
dev_loader = DataLoader(dev_ds, batch_size=32, shuffle=False,
                         num_workers=2, collate_fn=collate_fn)

mel_extractor = LogMelFilterBanks(
    n_fft=HP_FIXED["n_fft"], samplerate=HP_FIXED["samplerate"],
    hop_length=HP_FIXED["hop_length"], win_length=HP_FIXED["win_length"],
    n_mels=HP_FIXED["n_mels"],
).to(device)

print(f"Dev batches: {len(dev_loader)}")

In [ ]:
# Unigrams для BeamSearchDecoder — все слова из NUMBER_WORDS
# (ограничиваем beam-пространство русскими числовыми словами)
_unigrams: list[str] = []
for n in range(1000, 10001):  # небольшой subset для демо; увеличьте до 1_000_000 для production
    _unigrams.append(digits_to_words(n))
_unigrams = list(set(_unigrams))
print(f"Unigrams: {len(_unigrams)} unique word forms")

# Инициализируем оба декодера
beam_cfg = BeamSearchConfig(alpha=0.7, beta=1.0, beam_width=100,
                             hotwords=("тысяча", "тысячи", "тысяч"), hotword_weight=8.0)

if lm_binary.exists():
    beam_decoder = BeamSearchDecoder(vocab=vocab, kenlm_path=lm_binary,
                                      unigrams=_unigrams, config=beam_cfg)
else:
    print("LM binary not found — BeamSearchDecoder without KenLM")
    beam_decoder = BeamSearchDecoder(vocab=vocab, kenlm_path=None,
                                      unigrams=_unigrams, config=beam_cfg)

In [ ]:
if CKPT.exists():
    greedy_refs, greedy_hyps, greedy_spks = [], [], []
    beam_refs,   beam_hyps,   beam_spks   = [], [], []

    with torch.no_grad():
        for batch in dev_loader:
            audio = batch.audio.to(device)
            audio_lengths = batch.audio_lengths.to(device)
            mel = mel_extractor(audio)
            mel_lengths = (
                (audio_lengths + HP_FIXED["hop_length"] - 1) // HP_FIXED["hop_length"]
            ).clamp(max=mel.size(-1))
            out = model(mel, mel_lengths)

            hyps_greedy = greedy_decode(out.log_probs, out.output_lengths, vocab)
            hyps_beam   = beam_decoder.decode_batch(out.log_probs, out.output_lengths)

            greedy_refs.extend(batch.transcriptions)
            greedy_hyps.extend(hyps_greedy)
            greedy_spks.extend(batch.spk_ids)

            beam_refs.extend(batch.transcriptions)
            beam_hyps.extend(hyps_beam)
            beam_spks.extend(batch.spk_ids)

    ref_words_g = [digits_to_words(r) for r in greedy_refs]
    ref_words_b = [digits_to_words(r) for r in beam_refs]

    greedy_cer = compute_cer(ref_words_g, greedy_hyps)
    beam_cer   = compute_cer(ref_words_b, beam_hyps)

    greedy_spk_cer = compute_per_speaker_cer(ref_words_g, greedy_hyps, greedy_spks)
    beam_spk_cer   = compute_per_speaker_cer(ref_words_b, beam_hyps, beam_spks)

    print(f"Greedy CER: {greedy_cer:.4f}")
    print(f"Beam+LM CER: {beam_cer:.4f}")
else:
    print("Checkpoint not loaded — пропускаем декодирование.")
    greedy_cer = beam_cer = float("nan")
    greedy_spk_cer = beam_spk_cer = {}

## Шаг 4. Итог

In [ ]:
results = pd.DataFrame({
    "decoder": ["greedy", "beam+LM"],
    "cer": [greedy_cer, beam_cer],
    "max_spk_cer": [
        max(greedy_spk_cer.values()) if greedy_spk_cer else float("nan"),
        max(beam_spk_cer.values())   if beam_spk_cer   else float("nan"),
    ],
})
print(results.to_string(index=False))